# 11 — Lightweight Back-Trajectory Reconstruction


This notebook performs a simple kinematic back-trajectory reconstruction for selected overpass times around Newhaven. It is a transparent forensic screening tool, not a full dispersion model.


In [ ]:

SITE_ID = "NHV"
SITE_NAME = "Newhaven ERF"
LAT = 50.80208
LON = 0.04961
DATE_FROM = "2024-12-26"
DATE_TO   = "2025-01-05"
TARGET_TIME_UTC = None
HOURS_BACK = 24
STEP_HOURS = 1
SCENE_CATALOG_CSV = "outputs/s5p/NHV_no2_offl_scene_catalog.csv"
OUTPUT_DIR = "outputs/back_trajectory"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, math, hashlib
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:

def fetch_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "wind_speed_10m,wind_direction_10m",
        "timezone": "UTC",
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    j = r.json()
    h = j.get("hourly", {})
    return pd.DataFrame({
        "time_utc": pd.to_datetime(h.get("time", []), utc=True, errors="coerce"),
        "wind_speed_ms": h.get("wind_speed_10m", []),
        "wind_dir_deg": h.get("wind_direction_10m", []),
    })

wind = fetch_weather(LAT, LON, DATE_FROM, DATE_TO)
display(wind.head())


In [ ]:

catalog_path = Path(SCENE_CATALOG_CSV)
scene_df = pd.read_csv(catalog_path) if catalog_path.exists() else pd.DataFrame()
if not scene_df.empty and "start_utc" in scene_df.columns:
    scene_df["start_utc"] = pd.to_datetime(scene_df["start_utc"], utc=True, errors="coerce")
    scene_df = scene_df.dropna(subset=["start_utc"]).sort_values("start_utc").reset_index(drop=True)

if TARGET_TIME_UTC is None:
    target_ts = scene_df.iloc[0]["start_utc"] if not scene_df.empty else pd.Timestamp(f"{DATE_FROM}T12:00:00Z")
else:
    target_ts = pd.Timestamp(TARGET_TIME_UTC, tz="UTC")
print("Target time:", target_ts)


In [ ]:

KM_PER_DEG_LAT = 111.32
def km_per_deg_lon(lat_deg: float) -> float:
    return 111.32 * math.cos(math.radians(lat_deg))

def nearest_hour(ts, wind_df):
    idx = (wind_df["time_utc"] - ts).abs().idxmin()
    return wind_df.loc[idx]

def step_back(lat, lon, wind_speed_ms, wind_dir_deg, hours=1):
    distance_km = (wind_speed_ms * hours * 3600.0) / 1000.0
    bearing_deg = wind_dir_deg
    dlat = (distance_km * math.cos(math.radians(bearing_deg))) / KM_PER_DEG_LAT
    dlon = (distance_km * math.sin(math.radians(bearing_deg))) / km_per_deg_lon(lat)
    return lat + dlat, lon + dlon, distance_km

rows = []
cur_lat, cur_lon = LAT, LON
cur_time = target_ts
rows.append({"step": 0, "time_utc": cur_time, "lat": cur_lat, "lon": cur_lon, "wind_speed_ms": np.nan, "wind_dir_deg": np.nan, "distance_step_km": 0.0})
for i in range(1, HOURS_BACK + 1, STEP_HOURS):
    w = nearest_hour(cur_time, wind)
    ws = float(w["wind_speed_ms"]) if pd.notna(w["wind_speed_ms"]) else 0.0
    wd = float(w["wind_dir_deg"]) if pd.notna(w["wind_dir_deg"]) else 0.0
    next_lat, next_lon, dkm = step_back(cur_lat, cur_lon, ws, wd, hours=STEP_HOURS)
    cur_time = cur_time - pd.Timedelta(hours=STEP_HOURS)
    cur_lat, cur_lon = next_lat, next_lon
    rows.append({"step": i, "time_utc": cur_time, "lat": cur_lat, "lon": cur_lon, "wind_speed_ms": ws, "wind_dir_deg": wd, "distance_step_km": dkm})

traj = pd.DataFrame(rows)
display(traj.head(10))


In [ ]:

csv_path = Path(OUTPUT_DIR) / f"{SITE_ID}_back_trajectory.csv"
traj.to_csv(csv_path, index=False)

features = []
for _, r in traj.iterrows():
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(r["lon"]), float(r["lat"])]},
        "properties": {
            "step": int(r["step"]),
            "time_utc": str(r["time_utc"]),
            "wind_speed_ms": None if pd.isna(r["wind_speed_ms"]) else float(r["wind_speed_ms"]),
            "wind_dir_deg": None if pd.isna(r["wind_dir_deg"]) else float(r["wind_dir_deg"]),
        }
    })
line = {
    "type": "Feature",
    "geometry": {"type": "LineString", "coordinates": [[float(r["lon"]), float(r["lat"])] for _, r in traj.iterrows()]},
    "properties": {"site_id": SITE_ID, "site_name": SITE_NAME, "target_time_utc": str(target_ts)}
}
geojson = {"type": "FeatureCollection", "features": [line] + features}
geojson_path = Path(OUTPUT_DIR) / f"{SITE_ID}_back_trajectory.geojson"
geojson_path.write_text(json.dumps(geojson, indent=2), encoding="utf-8")

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(traj["lon"], traj["lat"], marker="o")
ax.scatter([LON], [LAT], marker="x", s=120)
ax.set_title(f"{SITE_NAME} lightweight back trajectory")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.tight_layout()
plot_path = Path(OUTPUT_DIR) / f"{SITE_ID}_back_trajectory.png"
fig.savefig(plot_path, dpi=150)
plt.show()

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "back_trajectory_reconstruction",
    "site_id": SITE_ID,
    "site_name": SITE_NAME,
    "lat": LAT,
    "lon": LON,
    "target_time_utc": str(target_ts),
    "hours_back": HOURS_BACK,
    "step_hours": STEP_HOURS,
    "rows_trajectory": int(len(traj)),
    "outputs": {"csv": str(csv_path), "geojson": str(geojson_path), "png": str(plot_path)},
    "notes": [
        "This is a lightweight kinematic back-trajectory, not a full atmospheric dispersion model.",
        "It is intended as a transparent forensic screening tool."
    ],
}
manifest_path = Path(OUTPUT_DIR) / f"{SITE_ID}_back_trajectory_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", csv_path)
print("Saved:", geojson_path)
print("Saved:", plot_path)
print("Saved:", manifest_path)
